# Retrain DETR — Improved Strawberry Disease Detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirmasoudGhorbani/hybrid-yolov9-detr-strawberry-disease/blob/main/notebooks/retrain_detr.ipynb)

This notebook retrains the DETR model with several improvements:

| Issue in Original | Fix |
|---|---|
| Generic `LABEL_0`, `LABEL_1` labels | Proper `id2label` mapping with disease names |
| No data augmentation | Horizontal flip, colour jitter, sharpness |
| Only 100 total epochs (30+20+50) | 150 epochs with cosine annealing LR |
| StepLR scheduler | Warmup + cosine annealing (smoother convergence) |
| No gradient clipping | Gradient clipping at 0.1 (training stability) |
| No best-model checkpoint | Auto-saves best model by validation loss |

**Runtime:** ~2-4 hours on Colab T4, ~1-2 hours on A100

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi

# Clone repo and install deps
!git clone https://github.com/AmirmasoudGhorbani/hybrid-yolov9-detr-strawberry-disease.git
%cd hybrid-yolov9-detr-strawberry-disease
!pip install -q -r requirements.txt

In [ ]:
# Mount Google Drive (your dataset lives here)
from google.colab import drive
drive.mount('/content/drive')

## 2. Verify Dataset Paths

Make sure these paths point to your dataset. Update `src/config.py` if they differ.

In [ ]:
import os
from src.config import DETR_TRAIN_PATH, DETR_VAL_PATH, DETR_TEST_PATH

for name, path in [('Train', DETR_TRAIN_PATH), ('Val', DETR_VAL_PATH), ('Test', DETR_TEST_PATH)]:
    exists = os.path.exists(path)
    count = len(os.listdir(path)) if exists else 0
    status = f'OK ({count} files)' if exists else 'NOT FOUND'
    print(f'{name}: {path} -> {status}')

if not os.path.exists(DETR_TRAIN_PATH):
    print('\n!! Update the paths in src/config.py to match your Google Drive !!')

## 3. Preview Augmentations

Let's see what the augmented training images look like.

In [ ]:
import matplotlib.pyplot as plt
from transformers import DetrImageProcessor
from src.utils import AugmentedCocoDetection
from src.config import DETR_BASE_MODEL

processor = DetrImageProcessor.from_pretrained(DETR_BASE_MODEL)
aug_dataset = AugmentedCocoDetection(DETR_TRAIN_PATH, processor, augment=True)
orig_dataset = AugmentedCocoDetection(DETR_TRAIN_PATH, processor, augment=False)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Top: Original — Bottom: Augmented', fontsize=14)
for i in range(4):
    idx = i * 10
    orig_pv, _ = orig_dataset[idx]
    aug_pv, _ = aug_dataset[idx]
    axes[0, i].imshow(orig_pv.permute(1, 2, 0).clip(0, 1))
    axes[0, i].axis('off')
    axes[1, i].imshow(aug_pv.permute(1, 2, 0).clip(0, 1))
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()
print(f'Dataset size: {len(aug_dataset)} images')

## 4. Train

This runs the improved training pipeline:
- **150 epochs** with warmup (10 epochs) + cosine annealing
- **Gradient clipping** at 0.1 for stability
- **Auto-checkpointing** saves the best model by validation loss
- **Data augmentation** on the training set

You can adjust these settings in the cell below.

In [ ]:
import os
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from transformers import DetrImageProcessor

from src.config import (
    DETR_BASE_MODEL, DETR_TRAIN_PATH, DETR_VAL_PATH,
    DETR_MODEL_V2, DETR_PROCESSOR_V2, ID2LABEL, LABEL2ID,
)
from src.utils import (
    AugmentedCocoDetection, CocoDetection,
    detr_collate_fn, print_final_metrics,
)
from src.retrain_detr import DetrWithCosineSchedule

# ── Settings (adjust these) ──────────────────────────────────────────
MAX_EPOCHS = 150       # increase for better results (200-300 for best)
BATCH_SIZE = 4         # reduce to 2 if you get OOM errors
LR = 1e-4              # learning rate for detection head
LR_BACKBONE = 1e-5     # learning rate for ResNet backbone
WARMUP_EPOCHS = 10     # linear warmup before cosine decay
NUM_WORKERS = 4        # dataloader workers
# ────────────────────────────────────────────────────────────────────

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['TORCH_USE_CUDA_DSA'] = '1'
torch.set_float32_matmul_precision('medium')

processor = DetrImageProcessor.from_pretrained(DETR_BASE_MODEL)

train_dataset = AugmentedCocoDetection(DETR_TRAIN_PATH, processor, augment=True)
val_dataset = CocoDetection(DETR_VAL_PATH, processor)
print(f'Training: {len(train_dataset)} images')
print(f'Validation: {len(val_dataset)} images')

collate = detr_collate_fn(processor)
train_dl = DataLoader(train_dataset, collate_fn=collate, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_dl = DataLoader(val_dataset, collate_fn=collate, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

model = DetrWithCosineSchedule(
    model_name_or_path=DETR_BASE_MODEL,
    lr=LR, lr_backbone=LR_BACKBONE, weight_decay=1e-4,
    warmup_epochs=WARMUP_EPOCHS, max_epochs=MAX_EPOCHS,
)

checkpoint_cb = ModelCheckpoint(
    monitor='validation_loss', mode='min', save_top_k=1,
    filename='detr-best-{epoch:02d}-{validation_loss:.4f}',
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    devices=1,
    accelerator='gpu',
    gradient_clip_val=0.1,
    log_every_n_steps=10,
    enable_progress_bar=True,
    callbacks=[checkpoint_cb, LearningRateMonitor('epoch')],
)

print('Starting training...')
trainer.fit(model, train_dl, val_dl)

## 5. Save the Best Model

In [ ]:
print_final_metrics(trainer)

# Load the best checkpoint
best_path = checkpoint_cb.best_model_path
print(f'Best checkpoint: {best_path}')
print(f'Best val loss: {checkpoint_cb.best_model_score:.4f}')

if best_path:
    best_model = DetrWithCosineSchedule.load_from_checkpoint(
        best_path,
        model_name_or_path=DETR_BASE_MODEL,
        lr=LR, lr_backbone=LR_BACKBONE, weight_decay=1e-4,
    )
else:
    best_model = model

# Set proper label mapping
best_model.model.config.id2label = ID2LABEL
best_model.model.config.label2id = LABEL2ID

# Save to Google Drive
best_model.model.save_pretrained(DETR_MODEL_V2)
processor.save_pretrained(DETR_PROCESSOR_V2)
print(f'\nModel saved to: {DETR_MODEL_V2}')
print(f'Processor saved to: {DETR_PROCESSOR_V2}')

## 6. Quick Test — Verify Detections

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image
from transformers import DetrForObjectDetection
from src.config import DETR_TEST_PATH, DISEASE_NAMES
from src.utils import CocoDetection, DEVICE

# Load the saved model
test_model = DetrForObjectDetection.from_pretrained(DETR_MODEL_V2).to(DEVICE)
test_model.eval()
test_processor = DetrImageProcessor.from_pretrained(DETR_PROCESSOR_V2)

# Check label mapping
print('Label mapping:')
for k, v in test_model.config.id2label.items():
    print(f'  {k}: {v}')

# Load test dataset
test_dataset = CocoDetection(DETR_TEST_PATH, test_processor, return_raw=True)

# Run inference on 6 random test images
fig, axes = plt.subplots(2, 3, figsize=(24, 16))
axes = axes.flat

for ax_idx in range(6):
    idx = random.randint(0, len(test_dataset) - 1)
    img, target = test_dataset[idx]
    image = Image.open(os.path.join(DETR_TEST_PATH, test_dataset.coco.loadImgs(test_dataset.ids[idx])[0]['file_name']))

    inputs = test_processor(images=image, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = test_model(pixel_values=inputs['pixel_values'])

    result = test_processor.post_process_object_detection(
        outputs, target_sizes=[(image.height, image.width)], threshold=0.3,
    )[0]

    ax = axes[ax_idx]
    ax.imshow(image)
    for score, label, box in zip(result['scores'], result['labels'], result['boxes']):
        xmin, ymin, xmax, ymax = box.cpu().numpy()
        name = test_model.config.id2label.get(label.item(), f'class_{label.item()}')
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin, edgecolor='lime', fill=False, linewidth=2))
        ax.text(xmin, ymin-4, f'{name} {score:.0%}', color='white', fontsize=9, fontweight='bold',
                bbox=dict(facecolor='green', alpha=0.8, pad=2))
    ax.set_title(f'Image {idx} — {len(result["scores"])} detections', fontsize=12)
    ax.axis('off')

plt.suptitle('Retrained DETR v2 — Test Set Predictions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Upload to HuggingFace

Once you're happy with the results, upload the new weights to your HuggingFace model repo.

In [ ]:
# Upload to HuggingFace Hub (run once, then comment out)
# !pip install -q huggingface-hub
# !huggingface-cli login

# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path=DETR_MODEL_V2,
#     repo_id='Amir-masoud-gh96/hybrid-yolov9-detr-strawberry',
#     path_in_repo='.',  # uploads config.json + model.safetensors to root
# )
# api.upload_folder(
#     folder_path=DETR_PROCESSOR_V2,
#     repo_id='Amir-masoud-gh96/hybrid-yolov9-detr-strawberry',
#     path_in_repo='.',
# )
# print('Uploaded to HuggingFace Hub!')